In [1]:
import csv
import os
import torchaudio
import torchaudio.transforms as T
import librosa
import soundfile as sf
os.environ['TRAINER_TELEMETRY'] = '0'

In [2]:
import certifi
#Trainer: Where the ✨️ happens.
# TrainingArgs: Defines the set of arguments of the Trainer.
from trainer import Trainer, TrainerArgs

# GlowTTSConfig: all model related values for training, validating and testing.
from TTS.tts.configs.tacotron2_config import Tacotron2Config

# BaseDatasetConfig: defines name, formatter and path of the dataset.
from TTS.tts.configs.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.tacotron2 import Tacotron2
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor

In [3]:
# we use the same path as this script as our training folder.
output_path = r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\TacotronProject'

In [4]:
new_data = []
database_path = os.path.join(output_path, 'RUSLAN')
count = 0
with open(os.path.join(database_path, 'metadata.txt'), 'r', encoding='utf-8') as f:
    lines = f.readlines()
    for line in lines:
        wav, text, transcription = line.split('|')
        #new_data.append({'wav': wav, 'transcription': transcription.rstrip('\n')})
        
        audio_path = output_path + '\\RUSLAN\\RUSLAN\\' + f"{wav}.wav"
        waveform, sr = librosa.load(audio_path, sr=None)
        #print(audio_path, sr)
        
        audio_len = len(waveform)/sr
        if audio_len <= 10 : #Исключить долгие файлы (их около 560)
            new_data.append({'wav': wav, 'transcription': transcription.rstrip('\n')})
        if sr != 16000:
            resample = librosa.resample(waveform, sr, 16000)
            sf.write(audio_path, resample, 16000)
            

In [5]:
with open(os.path.join(database_path, 'metadata.csv'), 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames =['wav', 'transcription'],delimiter='|')
    writer.writerows(new_data)

In [6]:
new_data[10:20]

[{'wav': '000000_RUSLAN',
  'transcription': "s' tr'i1vo0zhny4m chu0stva4m b'i1ru0s' ja0 za2 p'i1ro0 . ---"},
 {'wav': '000001_RUSLAN',
  'transcription': "ka1vo0 i1n't'i1r'i1su0ju4t pr'i1zna0n'i4ja4 l'i1t'i1ra0tu4rna4va4 n'i1u1da0chn'i4ka4 ? ----"},
 {'wav': '000002_RUSLAN',
  'transcription': "shto0 pa2u1chi0t'i4l'na4va4 vzno0j ji1vo0 i1spa1v'e0d'i4 ? ----"},
 {'wav': '000003_RUSLAN',
  'transcription': "da2 i1 zhy0z'n' ma1ja0 l'i1she0na4 vn'e0shn'i4va4 tra1g'i0za4 . ----"},
 {'wav': '000004_RUSLAN',
  'transcription': "ja0 a1ba1sl'u0tna4 zda1ro0f . ----"},
 {'wav': '000005_RUSLAN',
  'transcription': "u1 m'i1n'a0 je0z'd' l'u1b'a0sci4ja4 ro0d'n'a4 . ----"},
 {'wav': '000006_RUSLAN',
  'transcription': "mn'e0 fs'i1gda0 ga1to0vy4 pr'i1da1sta0v'i4t' ra1bo0tu4 , ---- ka1to0ra4ja4 a1b'i1sp'e0chi4t na1rma0l'na4ji4 b'i1a2la1g'i0chi4ska4ji4 su1sci1stva1va0va4ji4 . ---"},
 {'wav': '000007_RUSLAN',
  'transcription': "ma0la4 ta1vo0 , ---- ja0 a1bla0da4ju4 pr'i1i1mu0sci4sta4m'i4 . ----"},
 {'wa

In [7]:
# DEFINE DATASET CONFIG
# Set LJSpeech as our target dataset and define its path.
# You can also use a simple Dict to define the dataset and pass it to your custom formatter.
# dataset_config = BaseDatasetConfig(
#     formatter="ljspeech", meta_file_train="metadata.csv", path=os.path.join(output_path, r"C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\TacotronProject\RUSLAN")
# ) #r"C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\TacotronProject\LJSpeech-1.1")

dataset_config = BaseDatasetConfig(
     formatter="ruslan", meta_file_train="metadata.csv", path= r"C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\TacotronProject\RUSLAN")


In [8]:
# #DATASET_PATH = "/path/to/LJSpeech-1.1"   # сюда положен metadata.csv
# OUTPUT_PATH = "./tacotron2_out"          # где будут чекпоинты
# os.makedirs(OUTPUT_PATH, exist_ok=True)

In [9]:
# INITIALIZE THE TRAINING CONFIGURATION
# Configure the model. Every config class inherits the BaseTTSConfig.
config = Tacotron2Config(
    batch_size=32,
    eval_batch_size=16,
    num_loader_workers=0,
    num_eval_loader_workers=0,
    run_eval=True,
    test_delay_epochs=-1,
    epochs=1000,
    r=1,
    max_decoder_steps=1000,
    max_audio_len=160000,
    postnet_diff_spec_alpha=0,
    decoder_diff_spec_alpha=0,
    decoder_ssim_alpha=0,
    postnet_ssim_alpha=0,
    ga_alpha=0,
    optimizer='Adam',
    attention_type='dynamic_convolution',
    lr=0.00001,
    lr_scheduler=None,
    mixed_precision=False,
    #text_cleaner="phoneme_cleaners",
    use_phonemes=False,
    phoneme_language=None,
    phoneme_cache_path=os.path.join(output_path, "phoneme_cache"),
    print_step=25,
    print_eval=False,
    output_path=output_path,
    datasets=[dataset_config])

In [10]:
from TTS.tts.configs.shared_configs import CharactersConfig

In [11]:
symbols = ["t'", 'o0', 'm', 'n', 'y4', 'j', 'i4', 'a1', "l'", 'e0', 'f', 'h', 'l', 'd', 'a4', "s'", "n'", 'i1', 'a0', 'a2', 'z', 'b', 'sh', 'y0', 't', 'u0', 's', "k'", 'H', 'r', 'k', 'zh', "d'", "m'", "r'", "g'", 'ch', 'g', 'i0', "b'", 'v', "v'", "z'", 'y1', 'p', 'c', 'u1', "p'", 'u4', 'sc', 'o1', "h'", "f'", 'CH', 'C', 'o4', 'SC', 'e4', 'e1']
punctuation = ['.', ' ', ',','?', '!', '-']

In [12]:
config.audio.spec_gain = 1.0
config.audio.sample_rate = 16000
config.characters = CharactersConfig(characters=''.join(symbols),
                                     punctuations=''.join(punctuation),
                                     pad="<pad>",
                                     eos="<eos>",
                                     bos="<bos>",)

In [13]:
config.test_sentences = ["o0n i1ska0l ji1jo0 v g'i1l'i1nzhy0k'i4 , -- v ga0gra4h , - f so0chi4 . --",
     "na2 dru1go0j d'e0n' pa2 pr'i1je0z'd'i4 f so0chi4 ,  o0n ku1pa0ls'a4 u0tra4m vm mo0r'i4 ,  pa1to0m br'i0ls'a4 , - na1d'e0l chi0sta4ji4 b'i1l'jo0 , -- b'i1la1s'n'e0zhny4j k'i0t'i4l' , - pa1za0ftra4ka4l f sva1je0j ga2s't'i1n'i0cy4 na2 t'i1ra0s'i4 r'i1sta1ra0na4 , --- vy0p'i4l bu1ty0lku4 sha1mpa0nska4va4 , -- p'i0l ko0f'i4 s sha1rtr'e0za4m ,  n'i1 sp'i1sha0 vy0ku4r'i4l s'i1ga0ru4 .-- ",
     "va2zvra1t'a0s' v svo0j no0m'i4r ,  o0n l'o0k na2 d'i0va4n y1 vy0str'i4l'i4l s'i1b'e0 f v'i0sk'i4 y1z dvu0h r'i1vo0l'va4 . --",
     "ja0 n'i1 p'i1r'i1d che0m n'i1 a1sta1no0vl'u4s' , -  za2sci1sca0ja4 sva1ju0 che0s't' , -  che0s't' mu0zha4 i1 a1f'i1ce0ra4 ! ",
     "a1 za1che0m fs'o0 d'e0la4ji4ca4 na2 sv'e0t'i4 ? --- ",
     "ra0zv'i4 my0 pa2n'i1ma0ji4m shto0 n'i1bu0t' v na0shy4h pa2stu1pka0h ? ---",
     "v a1na0mn'i4z' su1p'i1rm'e0na4 by0l s'i1n't'i1z'i1ra1va0jny4j ka1m'p'ju0t'i4ra4m kr'i1pto0n'i4t . ---",
     "my0 fs'o0m a1tr'a0da4m sje0l'i4 fs'o0 , -- shto0 by0la4 vo0 fs'o0m la1g'e0r'i4 ,  i1 fs'o0 a1sta0l'i4s' da1vo0l'ny4 . ",
     "by0k tu1po0gu4p ,  tu1pa1gu0b'i4n'k'i4j by0cha4k , - u1 by1ka0 b'e0la4 gu1ba0 by1la0 tu1pa0 . ---",
     "ve0 d'e0n' tla1ka0shy4ji4s't'i4ca4 zhy1t'e0l'i4 t'i1na1chi0ta4la4 sla0v'a4t y1p'i0p' to0t'i4ka4 , | no1 n'i1 za2by1va0ju4t ta1kzhe0 i1 pro0 u1h'i1cy1la1cha0chi4l , --- k'i1ca0l'ka4t'a4la4ja4 i1 t'i1ska0t'i4pa4ku4 . ---",
     'vo0t a1no0 shto0 .  a1 o0n shto0 ? - da1 ty0 shto0 ! ---']

In [14]:
config.audio.mel_fmax=8000
config.audio.signal_norm=False


In [15]:
ap = AudioProcessor.init_from_config(config)
wav = ap.load_wav('C:\\Users\\ext-ananeva\\phon_clust\\gitlab\\ssl_phoneme_clusterizer\\TacotronProject\\RUSLAN\\RUSLAN\\012151_RUSLAN.wav')

mel = ap.melspectrogram(wav)
import matplotlib.pyplot as plt
import sys
plt.imshow(mel)
plt.colorbar()
plt.show()

 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:8000
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:1.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:45
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024


C:\Users\ext-ananeva\venv3.7\lib\site-packages\ipykernel_launcher.py:9: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  if __name__ == "__main__":


In [16]:
sf.info('C:\\Users\\ext-ananeva\\phon_clust\\gitlab\\ssl_phoneme_clusterizer\\TacotronProject\\RUSLAN\\RUSLAN\\012151_RUSLAN.wav').samplerate

16000

In [17]:
# INITIALIZE THE AUDIO PROCESSOR
# Audio processor is used for feature extraction and audio I/O.
# It mainly serves to the dataloader and the training loggers.
ap = AudioProcessor.init_from_config(config)

 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:8000
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:1.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:45
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024


In [18]:
# INITIALIZE THE TOKENIZER
# Tokenizer is used to convert text to sequences of token IDs.
# If characters are not defined in the config, default characters are passed to the config
tokenizer, config = TTSTokenizer.init_from_config(config)

In [19]:
# LOAD DATA SAMPLES
# Each sample is a list of ```[text, audio_file_path, speaker_name]```
# You can define your custom sample loader returning the list of samples.
# Or define your custom formatter and pass it to the `load_tts_samples`.
# Check `TTS.tts.datasets.load_tts_samples` for more details.
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

 | > Found 21650 files in C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\TacotronProject\RUSLAN


In [20]:
# INITIALIZE THE MODEL
# Models take a config object and a speaker manager as input
# Config defines the details of the model like the number of layers, the size of the embedding, etc.
# Speaker manager is used by multi-speaker models.
model = Tacotron2(config, ap, tokenizer, speaker_manager=None)

In [21]:
# import requests
# 
# url = "https://coqui.gateway.scarf.sh/trainer/training_run"
# 
# try:
#     r = requests.get(url)
#     print("Status code:", r.status_code)
# except requests.exceptions.SSLError as e:
#     print("SSL error:", e)
# except requests.exceptions.RequestException as e:
#     print("Other connection error:", e)

In [22]:
train_samples

[{'text': "chi1ta0ja4 , ---- ja0 n'i1za1m'e0tna4 u1snu0l . ---- pra1snu0ls'a4 vv'i1no0va4 dva0 no0chi4 . ---- pr'i1du1tr'e0n'i4j l'e0t'n'i4j su0mra4k za2l'i1va0l ko0mna4tu4 . ----\n",
  'audio_file': 'C:\\Users\\ext-ananeva\\phon_clust\\gitlab\\ssl_phoneme_clusterizer\\TacotronProject\\RUSLAN\\RUSLAN\\012230_RUSLAN.wav',
  'speaker_name': 'ruslan',
  'root_path': 'C:\\Users\\ext-ananeva\\phon_clust\\gitlab\\ssl_phoneme_clusterizer\\TacotronProject\\RUSLAN',
  'language': '',
  'audio_unique_name': '#RUSLAN\\012230_RUSLAN'},
 {'text': "pr'i1che0m d'i1rzha0l'i4 ji1vo0 n'i1 vs ha1lo0d'i4l'n'i4ka4h , ---- a1 m'e0zhdu4 a1ko0ny4m'i4 ra1ma0m'i4 . ----\n",
  'audio_file': 'C:\\Users\\ext-ananeva\\phon_clust\\gitlab\\ssl_phoneme_clusterizer\\TacotronProject\\RUSLAN\\RUSLAN\\014100_RUSLAN.wav',
  'speaker_name': 'ruslan',
  'root_path': 'C:\\Users\\ext-ananeva\\phon_clust\\gitlab\\ssl_phoneme_clusterizer\\TacotronProject\\RUSLAN',
  'language': '',
  'audio_unique_name': '#RUSLAN\\014100_RUSLAN'

In [23]:
# INITIALIZE THE TRAINER
# Trainer provides a generic API to train all the 🐸TTS models with all its perks like mixed-precision training,
# distributed training, etc.
trainer = Trainer(
    TrainerArgs(), config, output_path, model=model, train_samples=train_samples, eval_samples=eval_samples
)

 > Training Environment:
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 6
 | > Num. of Torch Threads: 6
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 > Start Tensorboard: tensorboard --logdir=C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\TacotronProject\run-December-12-2025_08+16PM-586346f

 > Model has 28089785 parameters


In [ ]:
# AND... 3,2,1... 🚀
trainer.fit()


 > EPOCH: 0/1000
 --> C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\TacotronProject\run-December-12-2025_08+16PM-586346f




> DataLoader initialization
| > Tokenizer:
	| > add_blank: False
	| > use_eos_bos: False
	| > use_phonemes: False
| > Number of instances : 21434



 > TRAINING (2025-12-12 20:16:40) 


 | > Preprocessing samples
 | > Max text length: 297
 | > Min text length: 16
 | > Avg text length: 128.5861715032192
 | 
 | > Max audio length: 159927.0
 | > Min audio length: 9709.0
 | > Avg audio length: 77026.11304469535
 | > Num. instances discarded samples: 0
 | > Batch group size: 0.
pr'i1kra0sna4 ! ----

 [!] Character '\n' not found in the vocabulary. Discarding it.


C:\Users\ext-ananeva\venv3.7\lib\site-packages\TTS\tts\models\tacotron2.py:334: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  alignment_lengths = mel_lengths // self.decoder.r

   --> STEP: 0/670 -- GLOBAL_STEP: 0
     | > decoder_loss: 4.50611  (4.50611)
     | > postnet_loss: 6.51955  (6.51955)
     | > stopnet_loss: 0.62451  (0.62451)
     | > loss: 3.38092  (3.38092)
     | > align_error: 0.90872  (0.90872)
     | > grad_norm: 2.03554  (2.03554)
     | > current_lr: 0.00001 
     | > step_time: 2.10970  (2.10968)
     | > loader_time: 0.07810  (0.07812)


   --> STEP: 25/670 -- GLOBAL_STEP: 25
     | > decoder_loss: 4.17273  (4.56321)
     | > postnet_loss

...


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.11024 (+0.00000)
     | > avg_decoder_loss: 0.60108 (+0.00000)
     | > avg_postnet_loss: 2.95712 (+0.00000)
     | > avg_stopnet_loss: 0.02144 (+0.00000)
     | > avg_loss: 0.91099 (+0.00000)
     | > avg_align_error: 0.97596 (+0.00000)

 > BEST MODEL : C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\TacotronProject\run-December-12-2025_08+16PM-586346f\best_model_670.pth

 > EPOCH: 1/1000
 --> C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\TacotronProject\run-December-12-2025_08+16PM-586346f




> DataLoader initialization
| > Tokenizer:
	| > add_blank: False
	| > use_eos_bos: False
	| > use_phonemes: False
	| > 2 not found characters:
	| > 

	| > |
| > Number of instances : 21434



 > TRAINING (2025-12-12 20:33:05) 


 | > Preprocessing samples
 | > Max text length: 297
 | > Min text length: 16
 | > Avg text length: 128.5861715032192
 | 
 | > Max audio length: 159927.0
 | > Min audio length: 9709.0
 | > Avg audio length: 77026.11304469535
 | > Num. instances discarded samples: 0
 | > Batch group size: 0.



   --> STEP: 5/670 -- GLOBAL_STEP: 675
     | > decoder_loss: 0.56931  (0.57424)
     | > postnet_loss: 1.67269  (1.67385)
     | > stopnet_loss: 0.04390  (0.04573)
     | > loss: 0.60441  (0.60775)
     | > align_error: 0.93765  (0.93646)
     | > grad_norm: 0.63909  (0.57138)
     | > current_lr: 0.00001 
     | > step_time: 0.36700  (0.25439)
     | > loader_time: 0.06280  (0.05980)


   --> STEP: 30/670 -- GLOBAL_STEP: 700
     | > decoder_loss: 0.49353  (0.53121)
     | > postnet_loss: 1.58523  (1.61778)
     | > stopnet_loss: 0.03318  (0.03932)
     | > loss: 0.55287  (0.57657)
     | > align_error: 0.95762  (0.94685)
     | > grad_norm: 0.37012  (0.46530)
     | > current_lr: 0.00001 
     | > step_time: 0.33780  (0.29349)
     | > loader_time: 0.08110  (0.06530)


   --> STEP: 55/670 -- GLOBAL_STEP: 725
     | > decoder_loss: 0.46867  (0.51356)
     | > postnet_loss: 1.57987  (1.60218)
     | > stopnet_loss: 0.02967  (0.03561)
     | > loss: 0.54181  (0.56454)
     | > align_e